# Unwrapping AI | Zero to Transformer
## Step 11 — Backpropagation: How Neural Networks Learn

**Creator:** Nitin K. Sethi  
**Goal:** Build a small neural network from scratch with NumPy, understand how errors travel backward, and visualize the learning process.

### Learning outcomes
1. Understand the forward pass and binary cross-entropy loss.
2. Calculate gradients using the chain rule (backpropagation).
3. Update weights using gradient descent.
4. Verify gradients numerically.
5. Visualize loss reduction and the learned decision boundary.

**Dataset:** Two Moons (synthetic binary classification). This is an educational example, not a production model.

## 1. The big picture

A neural network learns in a repeated cycle:

**Forward pass → Calculate loss → Backpropagate gradients → Update weights**

- The *forward pass* produces predictions.
- The *loss* measures how far predictions are from the true labels.
- *Backpropagation* calculates how each weight contributed to the error, using the chain rule.
- *Gradient descent* nudges weights in the direction that reduces the loss.

For this notebook, the network has **2 input features → 8 hidden neurons (tanh) → 1 sigmoid output**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

X, y = make_moons(n_samples=1000, noise=0.20, random_state=RANDOM_STATE)
y = y.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Positive class rate:", round(float(y_train.mean()), 3))

## 2. Visualize the dataset

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.ravel(), cmap="coolwarm", s=22, alpha=0.75)
plt.title("Training data: Two Moons")
plt.xlabel("Standardized feature 1")
plt.ylabel("Standardized feature 2")
plt.show()

## 3. Define the neural network

We initialize small random weights. The sigmoid function converts the output into a probability between 0 and 1. We clip probabilities slightly when calculating logarithms to avoid `log(0)`.

In [ ]:
n_inputs = 2
n_hidden = 8
n_outputs = 1

W1 = np.random.randn(n_inputs, n_hidden) * np.sqrt(1 / n_inputs)
b1 = np.zeros((1, n_hidden))
W2 = np.random.randn(n_hidden, n_outputs) * np.sqrt(1 / n_hidden)
b2 = np.zeros((1, n_outputs))

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    cache = (X, Z1, A1, Z2, A2)
    return A2, cache

def binary_cross_entropy(y_true, y_prob):
    eps = 1e-12
    p = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

probs, _ = forward(X_train, W1, b1, W2, b2)
print("Initial training loss:", round(float(binary_cross_entropy(y_train, probs)), 4))

## 4. Backpropagation: apply the chain rule

For a batch of `m` examples:

- Output layer: `dZ2 = (A2 - Y) / m`
- Hidden layer: `dZ1 = (dA1) * (1 - A1²)` because the derivative of `tanh(z)` is `1 - tanh(z)²`
- Weight gradients: `dW = previous_activations.T @ dZ`

The gradients tell us how to change each parameter to reduce the loss.

In [ ]:
def backward(y_true, cache, W2):
    X, Z1, A1, Z2, A2 = cache
    m = X.shape[0]

    dZ2 = (A2 - y_true) / m
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (1 - A1**2)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2

grads = backward(y_train, forward(X_train, W1, b1, W2, b2)[1], W2)
for name, grad in zip(["dW1", "db1", "dW2", "db2"], grads):
    print(name, grad.shape, "mean absolute gradient:", round(float(np.mean(np.abs(grad))), 6))

## 5. Train with gradient descent

In [ ]:
learning_rate = 0.1
epochs = 3000
loss_history = []

for epoch in range(epochs):
    train_probs, cache = forward(X_train, W1, b1, W2, b2)
    loss = binary_cross_entropy(y_train, train_probs)
    loss_history.append(loss)

    dW1, db1, dW2, db2 = backward(y_train, cache, W2)

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if epoch % 500 == 0:
        print(f"Epoch {epoch:4d} | loss = {loss:.4f}")

print(f"Final recorded training loss: {loss_history[-1]:.4f}")

## 6. Visualization: did the network learn?

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.title("Training loss across epochs")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
def plot_decision_boundary(X, y, title):
    x_min, x_max = X[:, 0].min() - 0.6, X[:, 0].max() + 0.6
    y_min, y_max = X[:, 1].min() - 0.6, X[:, 1].max() + 0.6
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                         np.linspace(y_min, y_max, 250))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_probs, _ = forward(grid, W1, b1, W2, b2)
    zz = (grid_probs >= 0.5).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, zz, alpha=0.22, cmap="coolwarm")
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", s=20, edgecolors="k", linewidths=0.2)
    plt.title(title)
    plt.xlabel("Standardized feature 1")
    plt.ylabel("Standardized feature 2")
    plt.show()

plot_decision_boundary(X_train, y_train, "Learned decision boundary — training data")

## 7. Evaluate on unseen test data

In [ ]:
test_probs, _ = forward(X_test, W1, b1, W2, b2)
test_pred = (test_probs >= 0.5).astype(int)

print("Test accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, test_pred, digits=3))

cm = confusion_matrix(y_test, test_pred)
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap="Blues")
plt.title("Test-set confusion matrix")
plt.show()

plot_decision_boundary(X_test, y_test, "Learned decision boundary — test data")

## 8. Gradient checking: is our backpropagation correct?

We compare one analytical gradient with a numerical approximation:

`(L(parameter + ε) - L(parameter - ε)) / (2ε)`

A small relative difference is evidence that the derivative implementation is correct. We check every parameter on a small batch to keep this step quick.

In [ ]:
def loss_for_params(Xb, yb, params):
    p, _ = forward(Xb, *params)
    return binary_cross_entropy(yb, p)

# Use a tiny independent parameter set for a clean numerical check.
rng = np.random.default_rng(7)
W1c = rng.normal(0, 0.2, (2, 3))
b1c = np.zeros((1, 3))
W2c = rng.normal(0, 0.2, (3, 1))
b2c = np.zeros((1, 1))
params = [W1c.copy(), b1c.copy(), W2c.copy(), b2c.copy()]
Xc, yc = X_train[:8], y_train[:8]

_, cache_c = forward(Xc, *params)
analytic = backward(yc, cache_c, params[2])

eps = 1e-5
max_abs_diff = 0.0
for p_idx, param in enumerate(params):
    for idx in np.ndindex(param.shape):
        plus = [p.copy() for p in params]
        minus = [p.copy() for p in params]
        plus[p_idx][idx] += eps
        minus[p_idx][idx] -= eps
        numerical = (loss_for_params(Xc, yc, plus) - loss_for_params(Xc, yc, minus)) / (2 * eps)
        difference = abs(numerical - analytic[p_idx][idx])
        max_abs_diff = max(max_abs_diff, difference)

print("Maximum absolute gradient difference:", f"{max_abs_diff:.10f}")
print("Gradient check:", "PASS" if max_abs_diff < 1e-6 else "REVIEW")

## 9. Experiment with the model

Change one setting at a time and rerun the training section:
- Hidden neurons: try 2, 4, 16, and 32.
- Learning rate: try 0.01, 0.1, and 0.5.
- Epochs: compare 500, 1500, and 3000.

Record the final training loss and test accuracy. Do not choose settings by repeatedly tuning against the test set; use a validation split for systematic model selection.

## 10. Key takeaways

- The forward pass makes predictions; the loss quantifies prediction error.
- Backpropagation applies the chain rule to calculate gradients layer by layer.
- Gradient descent uses those gradients to update weights and biases.
- Gradient checking compares analytical derivatives with numerical approximations.
- A falling training loss alone does not guarantee generalization; evaluate on unseen data.

### Step 11 completion checklist
- [ ] Run every cell in Google Colab from top to bottom.
- [ ] Confirm the loss curve generally decreases.
- [ ] Confirm the gradient check reports PASS.
- [ ] Record your test accuracy and save the notebook to GitHub.

**Next:** Step 12 — Transformers.